# 04 — Explicabilité SHAP
Interprétation globale et locale du modèle de production.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # racine ml/ -> import src.*
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
from src.features.pipeline import load_dataset, build_feature_matrix
b = joblib.load('../artifacts/best_model.joblib')
enc = joblib.load('../artifacts/encoders.joblib')
print('Modèle:', b['model_name'], '| type:', b['model_type'])

## Préparation des données de test scalées

In [ ]:
df = load_dataset().sample(300, random_state=42)
X,_ = build_feature_matrix(df, fit_encoders=False, encoders=enc)
X = X[b['feature_names']].copy()
X[b['numeric_cols']] = b['scaler'].transform(X[b['numeric_cols']])

## Importance globale (beeswarm)

In [ ]:
from src.explainability.shap_global import compute_global_importance
imp = compute_global_importance(b['raw_model'], X.to_numpy(), b['feature_names'],
    background=b.get('background'), model_type=b['model_type'])
pd.DataFrame(imp).head(12)

In [ ]:
from IPython.display import Image
Image('../artifacts/shap_global.png')

## Explication locale — lead HOT

In [ ]:
from src.explainability.shap_local import explain_lead
hot = df[(df.intention=='purchase-ready')&(df.sentiment=='positive')].head(1)
Xh,_ = build_feature_matrix(hot, fit_encoders=False, encoders=enc)
Xh = Xh[b['feature_names']].copy(); Xh[b['numeric_cols']] = b['scaler'].transform(Xh[b['numeric_cols']])
p = b['calibrated'].predict_proba(Xh.to_numpy())[0,1]
r = explain_lead(b['raw_model'], Xh.to_numpy()[0], b['feature_names'],
    int(round(p*100)), int(hot['lead_score'].iloc[0]),
    background=b.get('background'), model_type=b['model_type'])
print(r['explanation_text'])
pd.DataFrame(r['top_factors'])

## Explication locale — lead COLD

In [ ]:
cold = df[(df.intention=='information-gathering')&(df.sentiment=='negative')].head(1)
Xc,_ = build_feature_matrix(cold, fit_encoders=False, encoders=enc)
Xc = Xc[b['feature_names']].copy(); Xc[b['numeric_cols']] = b['scaler'].transform(Xc[b['numeric_cols']])
pc = b['calibrated'].predict_proba(Xc.to_numpy())[0,1]
rc = explain_lead(b['raw_model'], Xc.to_numpy()[0], b['feature_names'],
    int(round(pc*100)), int(cold['lead_score'].iloc[0]),
    background=b.get('background'), model_type=b['model_type'])
print(rc['explanation_text'])
pd.DataFrame(rc['top_factors'])

## Interprétation métier
Les facteurs dominants (intention d'achat, sentiment, urgence) correspondent au mécanisme réel de conversion : le modèle est cohérent et explicable pour un commercial.